# Data Exploration and Preprocessing

In [2]:
import pandas as pd
import duckdb
from pathlib import Path

## Download Instructions

If raw data is not saved as `data/raw/Books.jsonl.gz` and `data/raw/meta_Books.jsonl.gz`, follow the instructions in `README.md`

## Create Parquet Files If Needed

In [21]:
# Full files

if not Path("../data/raw/Books.parquet").is_file():
    duckdb.query("COPY (SELECT * FROM read_json_auto('../data/raw/Books.jsonl.gz')) TO '../data/raw/Books.parquet' (FORMAT PARQUET)")
if not Path("../data/raw/meta_Books.parquet").is_file():
    duckdb.query("COPY (SELECT * FROM read_json_auto('../data/raw/meta_Books.jsonl.gz', sample_size = 1000000)) TO '../data/raw/meta_Books.parquet' (FORMAT PARQUET)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [39]:
# Sample files

n_sample = 1000

con = duckdb.connect()

con.execute("""
SELECT setseed(0.42);
""")

# Create a temp table with the sampled rows
con.execute("""
CREATE TEMP TABLE sample AS
SELECT *
FROM read_json_auto('../data/raw/Books.jsonl.gz')
USING SAMPLE 1000 ROWS
""")

# Write sampled Books
con.execute("""
COPY sample
TO '../data/raw/Books_sample.parquet'
(FORMAT PARQUET)
""")

# Use sampled IDs to filter meta_Books
con.execute("""
COPY (
    SELECT f2.*
    FROM read_json_auto('../data/raw/meta_Books.jsonl.gz', sample_size = 1000000) f2
    INNER JOIN sample s
    ON f2.parent_asin = s.asin
) TO '../data/raw/meta_Books_sample.parquet'
(FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Dataset Overview

In [3]:
# fields in reviews file
duckdb.query("""
       DESCRIBE SELECT * FROM "../data/raw/Books.parquet";
""")

┌───────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │                                                  column_type                                                  │  null   │   key   │ default │  extra  │
│      varchar      │                                                    varchar                                                    │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ rating            │ DOUBLE                                                                                                        │ YES     │ NULL    │ NULL    │ NULL    │
│ title             │ VARCHAR                                                                                                     

In [4]:
# number of reviews
duckdb.query("""
       SELECT COUNT(*) FROM "../data/raw/Books.parquet";
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     29475453 │
└──────────────┘

In [5]:
# fields in meta books files
duckdb.query("""
       DESCRIBE SELECT * FROM "../data/raw/meta_Books.parquet";
""")

┌─────────────────┬───────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │                                column_type                                │  null   │   key   │ default │  extra  │
│     varchar     │                                  varchar                                  │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼───────────────────────────────────────────────────────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ main_category   │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ title           │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ subtitle        │ VARCHAR                                                                   │ YES     │ NULL    │ NULL    │ NULL    │
│ author          │ STRUCT(avatar VARCHAR, "name

In [6]:
# number of books
duckdb.query("""
       SELECT COUNT(*) FROM "../data/raw/meta_Books.parquet";
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      4448181 │
└──────────────┘

## Sample Records

### Review Records

In [7]:
books = pd.read_parquet("../data/raw/Books_sample.parquet",)
books.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4.0,Canada Reads Hits the Mark!,"From the description of the book, I thought it...",[],1553654021,1553654021,AFJPICFFQNQBIR4YNADEHNFVUAQQ,1357705896000,5,False
1,4.0,Huge timesaver filled with stuff I'd eat even ...,I'm a relatively new WW Online member (just a ...,[],1118116836,1118116836,AE3CYUANUQI2UJRDPCHFVTQSB3ZQ,1365620777000,17,True
2,3.0,Missing the hand itself,I was expecting more images of hands. There is...,[],0262018845,0262018845,AEY45G3XN75F4NHWF5E3LR6IHJDQ,1608234914287,0,True
3,2.0,Two Stars,Wasn't worth the money. The first story was en...,[],B00BSB2AE4,B00BSB2AE4,AEC5UJCQGSENH76NVUOH4CRXB4SA,1418931769000,0,True
4,5.0,Lots of Information,"Lots of Information, If you don't understand 1...",[],0964738627,0964738627,AFW7ZHB43ADEVVQ3SU7YV7TBMPEA,1438706821000,1,True


In [8]:
books.tail()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
995,5.0,Immersed in the world,I was mildly ingested in reading this book bec...,[],B082RYTGLW,B082RYTGLW,AFZFIOGSKOSSNK42SA2DRSR3IKHA,1585065829891,1,True
996,4.0,Beautiful poetry,"I read some Robert Browning in college, but I ...",[],B000JQUK96,B000JQUK96,AEAC4GSRKK25EFK7F4SWHXX2KSUA,1244776126000,2,False
997,5.0,Oh My ❤,I loved everything about this book. I can't g...,[],B01CFIEXIY,B01CFIEXIY,AGH4OHKOAWKIAY2BVG545G74WSOA,1562735069744,0,True
998,5.0,Hidden gem,How did I not know about this book before? Gr...,[],1631408976,1631408976,AHT6RMGKII2CXLJDYEDRPEE43BCA,1676007663517,0,True
999,5.0,Another Amazing Series from an Incredible Author,I absolutely LOVE this author! She is just so ...,[],B07W4KJRCK,B07W4KJRCK,AFHPV6VIOXK4L57NZYI72YYVBX2A,1595847422654,1,False


### Meta Book Records

In [9]:
meta_books = pd.read_parquet("../data/raw/meta_Books_sample.parquet")
meta_books.head()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Buy a Kindle,The Drowning Girl,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.1,279,"[A complex, haunting novel that explores a sch...","[Review, “Incisive, beautiful and as perfectly...",b'6.99',[{'large': 'https://m.media-amazon.com/images/...,[],Caitlin R. Kiernan (Author) Format: Kindle E...,"[Books, Literature & Fiction, Genre Fiction]","[(Publisher, b'""Ace; 1st edition (March 6, 201...",B006LU1T62,None
1,Books,Oracle of Mystical Moments,"Cards – February 5, 2018",None,4.8,1351,[Visionary collage artist Catrin Welz-Stein in...,"[About the Author, Catrin Welz-Stein uses digi...",b'17.2',[{'large': 'https://m.media-amazon.com/images/...,"[{'title': 'Flip Through & Review', 'url': 'ht...",Catrin Welz-Stein (Author),"[Books, Religion & Spirituality, New Age & Spi...","[(Publisher, b'""U.S. Games Systems, Inc. (Febr...",1572819200,None
2,Books,"Two Graves (Agent Pendergast Series, 12)","Hardcover – December 11, 2012",{'avatar': 'https://m.media-amazon.com/images/...,4.5,6136,"[After his wife, Helen, is brazenly abducted b...","[Review, ""The names Preston & Child on the cov...",b'19.91',[{'large': 'https://m.media-amazon.com/images/...,[],"Douglas Preston (Author), Lincoln Child (Author)","[Books, Literature & Fiction, Genre Fiction]","[(Publisher, b'""Grand Central Publishing; 1st ...",0446554995,None
3,Books,The Hunt for Jimmie Browne: An MIA Pilot in Wo...,"Hardcover – January 1, 2020",{'avatar': 'https://m.media-amazon.com/images/...,4.7,5,"[On Tuesday, November 17, 1942, aircraft CNAC ...","[Review, ""This book offers an interesting acco...",b'10.61',[{'large': 'https://m.media-amazon.com/images/...,[],Robert L. Willett (Author),"[Books, Biographies & Memoirs, Leaders & Notab...","[(Publisher, b'""POTOMAC BOOKS (January 1, 2020...",1640120254,None
4,Books,Natty & Mo,"Paperback – February 23, 2022",{'avatar': 'https://m.media-amazon.com/images/...,5.0,62,[A hedgehog with anxiety sabotages his own bir...,[],b'11.99',[{'large': 'https://m.media-amazon.com/images/...,[],"Susie Mendoza (Author), Alisa Angelone Ph.D (...","[Books, Children's Books, Growing Up & Facts o...","[(Publisher, b'""Independently published (Febru...",B09T662YFM,None


In [10]:
meta_books.tail()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
990,Books,She Believed She Could So She Did: Pink Notebo...,"Paperback – August 17, 2016",{'avatar': 'https://m.media-amazon.com/images/...,4.6,492,[This bright pink notebook features the inspir...,[],b'6.66',[],[],Joy Tree Journals (Author),[],"[(Publisher, b'""CreateSpace Independent Publis...",153713261X,None
991,Books,Acadian Tapestry Address Book (Address Books),"Spiral-bound – January 1, 2009",{'avatar': 'https://m.media-amazon.com/images/...,4.4,915,[Keeping track of family and friends is easy w...,[],b'8.99',[],[],"Peter Pauper Press (Author, Editor)","[Books, Stationery, Journals & Notebooks, Jour...","[(Publisher, b'""Peter Pauper Press; Spi editio...",1593592248,None
992,Books,Bunny Brown and His Sister Sue at the Summer C...,"Hardcover – January 1, 1931",None,5.0,2,[],[],"b'""from 49.95""'",[],[],Laura Lee Hope (Author),"[Books, History, Historical Study & Educationa...","[(Publisher, b'""Grosset & Dunlap; Not Stated e...",B000PLZ9FA,None
993,Books,Toddler Books About Sports Cars Wordless Pictu...,"Paperback – February 4, 2022",{'avatar': 'https://m.media-amazon.com/images/...,3.6,4,[Introducing a unique new series of wordless p...,[],b'10.91',[],[],Busy Hands Books (Author),"[Books, Children's Books, Arts, Music & Photog...","[(Publisher, b'""Independently published (Febru...",B09RLXXVWV,None
994,Books,Liberating Duality with Wisdom Display: The Ei...,"Paperback – January 1, 2013",None,4.7,8,[Guru Padmasambhava is generally referred to a...,[],b'22.0',[],[],"Khenchen Palden Sherab Rinpoche (Author), Khe...","[Books, Religion & Spirituality, Buddhism]","[(Publisher, b'""Dharma Samudra (January 1, 201...",0983407428,None


In [27]:
str(meta_books.loc[0, "author"]).split(sep = ",")[1].split(sep = ": ")[1]

"'Caitlin R. Kiernan'"

In [47]:
bool(meta_books.loc[0, "author"])

True

In [39]:
str(meta_books.loc[7, "author"]).split(sep = ",")[1].split(sep = ": ")[1]

"'Pat Pattison'"

## Selection of Fields and Justification

From the review data, we will keep only asin, text, and rating. We think this in

Which fields we are keeping and removing

- Book reviews
  - KEEP
    - rating: this is helpful information to display when looking up books
    - text: this is helpful information to display when looking up books
    - asin: to connect the review to the book data
  - REMOVE
    - title: this is already in meta data
    - images: we don't be displaying images in our simple search app
    - user_id: this isn't relevant information for our search app
    - timestamp: this isn't relevant information for our search app
    - helpful_vote: this could be helpful information to use in the future but we are just keeping it simple for now
    - verified_purchase: this could be helpful information to use in the future but we are just keeping it simple for now
- Book meta data
  - KEEP
    - title: key information
    - author: keep author name but remove image and bio, the image we won't be using in this simple app and the bio might get confused with book info in dense encodings (searching for a book about Ireland might return books that are not from Ireland but are by an Irish writer)
    - average_rating: helpful information to view
    - rating_number: needed to context for average rating
    - features: this is the feature that gives the best book description, one of the most important features for searching
    - price: helpful information to view
    - parent_asin: needed to find the relevant reviews
    - categories: give genre information, this is also helpful for search
  - REMOVE
    - main_category: inconsistent information, generally says the book format which isn't important for this book search app
    - subtitle: inconsistent information, also information on format
    - description: mix of reviews and description info, not very consistent or of high quality
    - images: not needed for this search app
    - video: not needed for this search app
    - store: inconsistent data and not needed for the search app
    - details: publisher information, not needed for this search app
    - bought_together: amazon purchase information, not needed for this search app


## Text Preprocessing Decisions

## Preprocess and Save Data

In [69]:
# Save only relevant book review features
con = duckdb.connect()
con.execute("""
    COPY (
        SELECT text, rating, asin
        FROM "../data/raw/Books.parquet"
    ) TO "../data/processed/Books_processed.parquet"
    (FORMAT PARQUET)
    """
)

# Relevant meta data with some cleaning
con.execute("""
    COPY (
        SELECT 
            title,
            COALESCE(author.name, '') AS author,
            array_to_string(features, ', ') AS features,
            array_to_string(categories, ', ') AS categories,
            average_rating,
            rating_number, 
            price, 
            parent_asin
        FROM "../data/raw/meta_Books.parquet"
    ) TO "../data/processed/meta_Books_processed.parquet"
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
check_reviews = con.execute("""
    SELECT *
    FROM "../data/processed/Books_processed.parquet"
    LIMIT 10
    """
)

display(check_reviews.df())

,text,rating,asin
0,It is definitely not a watercolor book. The p...,1.0,B09BGPFTDB
1,Updated: after first book arrived very damaged...,5.0,0593235657
2,I bought it for the bag on the front so it pai...,5.0,1782490671
3,Updated: after 1st arrived damaged the replace...,5.0,0593138228
4,I love this book! The patterns are lovely. I ...,5.0,0823098079
5,Missing the sketch pad. Even worse I realized ...,1.0,1631591290
6,Seriously one of only a few books they I have ...,4.0,1640210148
7,I love this book. I was not blessed with arti...,5.0,1784881953
8,I really wanted to like this book bc I have he...,3.0,1645671127
9,Every page has a crease running the entire len...,1.0,1780671067


In [70]:
check_meta = con.execute("""
    SELECT *
    FROM "../data/processed/meta_Books_processed.parquet"
    LIMIT 10
    """
)

display(check_meta.df())

,title,author,features,categories,average_rating,rating_number,price,parent_asin
0,Chaucer,Peter Ackroyd,,"Books, Literature & Fiction, History & Criticism",4.5,29,8.23,0701169850
1,Notes from a Kidwatcher,Yetta M. Goodman,Contains 23 selected articles by this influent...,"Books, Reference, Words, Language & Grammar",5.0,1,3.52,0435088688
2,Service: A Navy SEAL at War,Marcus Luttrell,"Marcus Luttrell, author of the #1 bestseller, ...","Books, Biographies & Memoirs, Leaders & Notabl...",4.7,3421,17.17,0316185361
3,Monstrous Stories #4: The Day the Mice Stood S...,,"Funny, light-hearted monster stories that are ...","Books, Children's Books, Science Fiction & Fan...",4.4,40,7.43,0545425573
4,Parker & Knight,Donald Wells,"From REMINGTON KANE, the author of The Taken! ...","Books, Mystery, Thriller & Suspense, Thrillers...",4.5,381,0.0,B00KFOP3RG
5,Writings from a Black Woman Living in the Land...,,Take a step into the modern perspective of a y...,"Books, Arts & Photography, History & Criticism",5.0,5,4.05,B09PHG4FQ8
6,Child Development: A Practitioner's Guide:2nd ...,,"Child Development, Second EditionDouglas Davies","Books, Parenting & Relationships, Parenting",5.0,2,10.68,B0086HQWC4
7,Make: Electronics: Learning Through Discovery,Charles Platt,"""This is teaching at its best!"", Hans Camenzin...","Books, Engineering & Transportation, Engineering",4.7,1366,13.43,1680450263
8,Reunion: The Children of Lauderdale Park,,"1940-Sadie, Jacob, Seth, and Hattie Lauderdale...","Books, Literature & Fiction, Genre Fiction",4.9,12,14.0,1694621731
9,Four Centuries of American Education,David Barton,"For four centuries, religion, morality, and kn...","Books, Education & Teaching, Schools & Teaching",4.8,133,6.99,1932225323


## Save Processed Data

In [ ]:
# Note, save as that fancy format to use with llms??